# Compute Mean and Standard Deviation
Script to compute the mean and standard deviation of the dataset pixels.

In [1]:
"""
Computes per-channel mean and std of the CXR8 training set
after CLAHE and before normalization.
Run once, then paste the output into your training config.
"""
import os
import glob
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm
from sklearn.model_selection import train_test_split

from dataset import (
    CLIP_LIMIT, TILE_GRID_SIZE,
    CLAHETransform, normalize_cxr_image, ALL_CLASSES
)
from sklearn.preprocessing import MultiLabelBinarizer

### Paths — match your training script ###
METADATA_CSV_PATH = "../chest_xray_dataset/CXR8/Data_Entry_2017_v2020.csv"
IMAGE_ROOT        = "../chest_xray_dataset/CXR8/images"

BATCH_SIZE  = 8
NUM_WORKERS = 0
IMG_SIZE    = 256

def worker_init_fn(worker_id):
    cv2.setNumThreads(0)

class StatsDataset(Dataset):
    """Minimal dataset: resize + CLAHE only, no normalization."""
    def __init__(self, df, idx_array, lookup):
        self.df     = df.iloc[idx_array].reset_index(drop=True)
        self.lookup = lookup
        self.tf = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            CLAHETransform(
                clip_limit=CLIP_LIMIT,
                tile_grid_size=(TILE_GRID_SIZE, TILE_GRID_SIZE)
            ),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        while True:
            fname = self.df.loc[i, "Image Index"]
            img   = Image.open(self.lookup[fname])
            arr   = normalize_cxr_image(img)
            if arr is None:
                i = (i + 1) % len(self.df)
                continue
            arr = np.stack([arr, arr, arr], axis=-1)
            img = Image.fromarray(arr)
            return self.tf(img)


def compute_mean_std(loader):
    """Welford online algorithm — numerically stable, single pass."""
    n      = 0
    mean   = torch.zeros(3)
    M2     = torch.zeros(3)  # sum of squared deviations

    for imgs in tqdm(loader, desc="Computing stats"):
        # imgs: (B, C, H, W)
        B, C, H, W = imgs.shape
        pixels = imgs.permute(1, 0, 2, 3).reshape(C, -1)  # (C, B*H*W)

        for px in pixels.t():                              # iterate pixels
            n    += 1
            delta = px - mean
            mean += delta / n
            M2   += delta * (px - mean)

    std = (M2 / (n - 1)).sqrt()
    return mean, std


def compute_mean_std_fast(loader):
    """
    Faster batch-level version (slight numerical difference vs Welford
    but fine for normalization constants).
    """
    mean_acc = torch.zeros(3)
    var_acc  = torch.zeros(3)
    n_pixels = 0

    for imgs in tqdm(loader, desc="Computing stats"):
        B, C, H, W = imgs.shape
        n = B * H * W
        # per-channel mean and var over this batch
        pixels     = imgs.permute(1, 0, 2, 3).reshape(C, -1).float()
        mean_acc  += pixels.mean(dim=1) * n
        var_acc   += pixels.var(dim=1, unbiased=False) * n
        n_pixels  += n

    mean = mean_acc / n_pixels
    std  = (var_acc  / n_pixels).sqrt()
    return mean, std


if __name__ == "__main__":
    # Load metadata
    df = pd.read_csv(METADATA_CSV_PATH)
    df = df[["Image Index", "Finding Labels", "View Position"]].copy()
    df = df[df["View Position"].isin(["PA", "AP"])].reset_index(drop=True)
    df["labels"] = df["Finding Labels"].str.split("|")

    # Image lookup
    all_png    = glob.glob(os.path.join(IMAGE_ROOT, "**", "*.png"), recursive=True)
    path_lookup = {os.path.basename(p): p for p in all_png}

    mask = df["Image Index"].isin(path_lookup)
    df   = df[mask].reset_index(drop=True)

    # Same split as training so stats are train-only (no val leakage)
    indices = np.arange(len(df))
    train_idx, _ = train_test_split(indices, test_size=0.15, random_state=42)

    ds = StatsDataset(df, train_idx, path_lookup)
    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        worker_init_fn=worker_init_fn,  
    )

    print(f"Computing over {len(ds)} training images at {IMG_SIZE}x{IMG_SIZE}...")
    mean, std = compute_mean_std_fast(loader)

    print("\n--- Results ---")
    print(f"Mean: [{mean[0]:.4f}, {mean[1]:.4f}, {mean[2]:.4f}]")
    print(f"Std:  [{std[0]:.4f}, {std[1]:.4f}, {std[2]:.4f}]")
    print("\nPaste into dataset.py:")
    print(f"CXR8_MEAN = [{mean[0]:.4f}, {mean[1]:.4f}, {mean[2]:.4f}]")
    print(f"CXR8_STD  = [{std[0]:.4f}, {std[1]:.4f}, {std[2]:.4f}]")
    print("\n(All 3 channels should be near-identical since images are grayscale-stacked.)")

Computing over 95302 training images at 256x256...


Computing stats: 100%|██████████| 11913/11913 [50:48<00:00,  3.91it/s]


--- Results ---
Mean: [0.5249, 0.5249, 0.5249]
Std:  [0.2622, 0.2622, 0.2622]

Paste into dataset.py:
CXR8_MEAN = [0.5249, 0.5249, 0.5249]
CXR8_STD  = [0.2622, 0.2622, 0.2622]

(All 3 channels should be near-identical since images are grayscale-stacked.)


In [ ]:



import numpy as np
from PIL import Image
from tqdm import tqdm
import os
import glob

# Build image-path index
IMAGE_ROOT = r"C:\Users\nick\computing_for_health_and_medicine\chest_xray_dataset\CXR8\images"



# Recursively find every PNG once and build a filename -> full-path dict
all_png = glob.glob(os.path.join(IMAGE_ROOT, "**", "*.png"), recursive=True)
path_lookup = {os.path.basename(p): p for p in all_png}
print(f"Found {len(path_lookup):,} images on disk.")



def compute_mean_std(image_paths):
    n_pixels = 0
    channel_sum = 0.0
    channel_sq_sum = 0.0

    for p in tqdm(image_paths):
        
        img = Image.open(p).convert("RGB")
        arr = np.array(img) / 255.0   # scale to [0,1]
        arr = arr.reshape(-1, 3)      # flatten pixels

        channel_sum += arr.sum(axis=0)
        channel_sq_sum += (arr ** 2).sum(axis=0)
        n_pixels += arr.shape[0]

    mean = channel_sum / n_pixels
    std = np.sqrt(channel_sq_sum / n_pixels - mean ** 2)
    return mean, std




image_paths = list(path_lookup.values())
mean, std = compute_mean_std(image_paths)
print("Mean:", mean)
print("Std:", std)